# Faster R-CNN 訓練與推論

課程提交版本的公開整理副本。所有歷史輸出、附件及非必要 metadata 已移除。
先閱讀上層 README 的資料需求、已知問題與重現限制；不要直接 Run All。


# Object detection on medical image


In [ ]:
# import libraries

# basic
#import warnings
#warnings.filterwarnings('ignore')

import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import random
import numpy as np
import pandas as pd
import math
from tqdm.notebook import tqdm

# visualization
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# PyTorch
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection import FasterRCNN
from torchvision.transforms import v2
from torchvision import tv_tensors
from torchvision.tv_tensors import BoundingBoxes

# object detection
#!pip install pycocotools
import pycocotools
from pycocotools.coco import COCO

In [ ]:
#!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/engine.py
#!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/utils.py
#!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_utils.py
#!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/coco_eval.py
#!wget https://raw.githubusercontent.com/pytorch/vision/main/references/detection/transforms.py
    
from engine import evaluate

In [ ]:
## TODO: Prepare your own information
class config:
    
    ## roots for training & valid
    root = "hwk05_data/output"
    info_root = "hwk05_data"
    save_root = "hwk05_data"

    ## for test images
    test_root = 'hwk05_data/output/test'
    info_root_test = 'hwk05_data/test'
    
    num_classes = 8 #(for fasterrcnn: background + # of classes): 1+7=8
    batch_size = 8
    epochs = 20
    weight_decay = 0.005
    lr = 0.001
    momentum = 0.8
    seed = 42
    workers = 0
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
def seed_everything(seed):
   
    random.seed(seed) # Set Python random seed    
    np.random.seed(seed) # Set NumPy random seed    
    torch.manual_seed(seed) # Set PyTorch random seed for CPU and GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Set PyTorch deterministic operations for cudnn backend
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    
seed_everything(config.seed)

# Read data information


In [ ]:
annfile = config.info_root + "/train.json"
coco = COCO(annfile)
coco.cats

## Images


In [ ]:
coco.loadImgs(0)

## Annotations


In [ ]:
ann_ids = coco.getAnnIds(imgIds = 0)
coco.loadAnns(ann_ids)

In [ ]:
ann_ids = coco.getAnnIds(imgIds = 155)
coco.loadAnns(ann_ids)

In [ ]:
del coco

# Data augmentation


In [ ]:
## TO DO: Augmentation?
from torchvision import tv_tensors
from torchvision.transforms import v2

class medTransform:
    def __init__(self, train=False):    
        if train:
            self.transforms = v2.Compose(
                [
                    v2.ToImage(),  # 適用 PIL 圖片
                    #v2.RandomHorizontalFlip(p=0.5),  # 隨機水平翻轉
                    #v2.RandomRotation(degrees=15),   # 隨機旋轉 ±15度
                    #v2.ColorJitter(brightness=0.2, contrast=0.2),  # 顏色增強
                    v2.Resize((512, 512)),  # 縮放影像至固定大小
                    v2.ToDtype(torch.float32, scale=True),
                ]
            )
        else:
            self.transforms = v2.Compose(
                [
                    v2.ToImage(),
                    v2.Resize((512, 512)),
                    v2.ToDtype(torch.float32, scale=True),
                ]
            )       

    def __call__(self, x, bboxes):
        if isinstance(x, torch.Tensor):
            height, width = x.shape[-2], x.shape[-1]  # (C, H, W) 格式
        else:  # PIL Image
            width, height = x.size
        
        # 當 bounding box 為空時，避免維度錯誤
        if bboxes is None or len(bboxes) == 0:
            return self.transforms(x), bboxes
        
        # 將 bounding box 轉為 torchvision 支援格式
        bboxes = tv_tensors.BoundingBoxes(bboxes, format="XYWH", canvas_size=(height, width))
        
        # 使用 torchvision transforms 處理影像和 bounding box
        transformed_image, transformed_bboxes = self.transforms(x, bboxes)

        return transformed_image, transformed_bboxes



# Dataset


In [ ]:
class MedDataset(Dataset):
    
    def __init__(self, root, info_root, split, transforms = None):
        self.split = split
        self.root = root
        self.info_root = info_root
        self.transforms = transforms
        self.coco = COCO(os.path.join(self.info_root, f"{self.split}.json"))
        self.ids = list(sorted(self.coco.imgs.keys()))
    
    def get_image(self, img_id: int):
        image_path = os.path.join(self.root,self.coco.loadImgs(img_id)[0]['file_name'])
        image = Image.open(image_path).convert("RGB")
        return image

    def get_annotation(self, img_id: int):        
        return self.coco.loadAnns(self.coco.getAnnIds(img_id))
          
    def __getitem__(self, index):
        normal = False
        img_id = self.ids[index]
        image = self.get_image(img_id) 
        annotation = self.get_annotation(img_id)
        
        bboxes = [a['bbox']  for a in annotation]
        category_ids = [a['category_id']  for a in annotation]
        if bboxes == []:
            normal = True     
            
        if self.transforms:
            image, bboxes = self.transforms(image, bboxes)     
            
        def reformat_bboxes(boxes):
            return [[val[0], val[1], val[0] + val[2], val[1] + val[3]] for val in boxes]
        
        if normal != True:
            ## Recall that the original format is COCO
            bboxes = reformat_bboxes(bboxes)   

        def create_target(bboxes, normal):
            if normal:
                return {
                    'boxes': torch.zeros((0, 4), dtype=torch.float32),  # Empty boxes
                    'labels': torch.tensor(category_ids, dtype=torch.int64),
                    'image_id': img_id,
                    'area': torch.zeros((0,), dtype=torch.float32),  # Empty areas
                    'iscrowd': torch.zeros((0,), dtype=torch.int64),  # Empty tensor for iscrowd
                }
            else:
            
                return {
                    'boxes': torch.tensor(bboxes),
                    'labels': torch.tensor(category_ids, dtype=torch.int64),
                    'image_id': img_id,
                    'area': torch.tensor([(bbox[2] - bbox[0]) * (bbox[3] - bbox[1]) for bbox in bboxes], dtype=torch.float32),
                    'iscrowd': torch.tensor([a['iscrowd'] for a in annotation], dtype=torch.int64)
                }
       
        targets = create_target(bboxes,normal)
        return image, targets
        
    def __len__(self):
        return len(self.ids)

## Collate_fn


In [ ]:
def collate_fn(batch: list[torch.tensor, dict]):
    return tuple(zip(*batch))

In [ ]:
def plot_image_with_boxes(image_tensor, boxes_dict):
    image_np = image_tensor.permute(1, 2, 0).numpy()
    fig, ax = plt.subplots(1)
    # Display the image
    ax.imshow(image_np)
    for box in boxes_dict['boxes']:
        # Extract coordinates (x0, y0, x1, y1)
        x0, y0, x1, y1 = box
        # Calculate the height as (y0 - y1) since y0 is the top and y1 is the bottom
        height = y1 - y0
        # Create a rectangle patch with (x0, y0) as the top-left corner
        rect = patches.Rectangle((x0, y0), x1 - x0, height, linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)
    plt.show()

In [ ]:
train_dataset = MedDataset(root = config.root, info_root = config.info_root, split = "train", transforms = medTransform(train=True))
train_loader = DataLoader(train_dataset, batch_size = config.batch_size, shuffle = True,num_workers=4, collate_fn = collate_fn)

In [ ]:
a,b = train_dataset.__getitem__(1)
plot_image_with_boxes(a,b)

In [ ]:
a,b = train_dataset.__getitem__(155)
print(b)
plot_image_with_boxes(a,b)

# Model: Faster R-CNN


In [ ]:
def fasterrcnn(num_classes):
    model = models.detection.fasterrcnn_resnet50_fpn(weights='COCO_V1') 
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = None
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

In [ ]:
model = fasterrcnn(config.num_classes)

# Training


In [ ]:
def train_one_epoch(model, train_loader, optimizer, epoch, device):
    model.train()
    
    train_loss = []
    train_loss_dict = []
    
    lr_scheduler = None

    for images, targets in tqdm(train_loader):
        images = [image.to(device) for image in images]
        targets = [{k: (torch.tensor(v,device=device) if not isinstance(v, torch.Tensor) else v.to(device)) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        batch_loss_value = losses.item()
        batch_loss_dict = {k: v.item() for k, v in loss_dict.items()}

        train_loss.append(batch_loss_value)
        train_loss_dict.append(batch_loss_dict)
      
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
    
        if lr_scheduler is not None:
            lr_scheduler.step()
        
    train_loss = np.mean(train_loss)    
    train_loss_dict = pd.DataFrame(train_loss_dict).mean()
    train_loss_classifier = train_loss_dict.loss_classifier
    train_loss_box_reg = train_loss_dict.loss_box_reg
    train_loss_rpn_box_reg = train_loss_dict.loss_rpn_box_reg
    train_loss_objectness = train_loss_dict.loss_objectness

    return train_loss, train_loss_classifier, train_loss_box_reg, train_loss_rpn_box_reg, train_loss_objectness

# Validation


In [ ]:
def validation(model, val_loader, device):
    model.train()
    #model.eval()
    for m in model.modules():
        if isinstance(m, torchvision.ops.Conv2dNormActivation):
            m.eval()
        if isinstance(m, torchvision.ops.FrozenBatchNorm2d):
            m.eval()
        if isinstance(m, torch.nn.BatchNorm2d):
            m.eval()
    val_loss = []
    val_loss_dict = []
    with torch.no_grad():
        for images, targets in tqdm(val_loader):
            images = [image.to(device) for image in images]
            targets = [{k: (torch.tensor(v,device=device) if not isinstance(v, torch.Tensor) else v.to(device)) for k, v in t.items()} for t in targets]

            loss = model(images, targets)
            total_loss = sum(l for l in loss.values())
            
            loss_value = total_loss.item()
            loss_dict = {k: v.item() for k, v in loss.items()}
            
            val_loss.append(loss_value)
            val_loss_dict.append(loss_dict)
    
    val_loss = np.mean(val_loss)
    
    val_loss_dict = pd.DataFrame(val_loss_dict).mean()
    val_loss_classifier = val_loss_dict.loss_classifier
    val_loss_box_reg = val_loss_dict.loss_box_reg
    val_loss_rpn_box_reg = val_loss_dict.loss_rpn_box_reg
    val_loss_objectness = val_loss_dict.loss_objectness
    
    return val_loss, val_loss_classifier, val_loss_box_reg, val_loss_rpn_box_reg, val_loss_objectness

In [ ]:
del model
del train_dataset, train_loader

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Main


In [ ]:
def main():
    
    seed_everything(config.seed)    
    g = torch.Generator()
    g.manual_seed(config.seed)
     
    train_dataset = MedDataset(root = config.root, info_root = config.info_root, split = "train", transforms = medTransform(train=True))
    val_dataset = MedDataset(root = config.root, info_root = config.info_root, split = "val",  transforms = medTransform(train=False))

    train_loader = DataLoader(train_dataset, batch_size = config.batch_size, shuffle = True,
                              num_workers=config.workers, collate_fn = collate_fn,pin_memory=True
                             )
    val_loader = DataLoader(val_dataset, batch_size = config.batch_size, shuffle = False,
                            num_workers=config.workers, worker_init_fn=seed_worker,
                            generator=g, collate_fn = collate_fn,pin_memory=True
                           )

    
    device = config.device
    model =  fasterrcnn(num_classes = config.num_classes)
    model.to(device)
    
    parameters = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.SGD(parameters, lr = config.lr, momentum = config.momentum, nesterov = True, weight_decay = config.weight_decay)
    best_val_loss = float("inf")
    best_map50 = 0.0
    history = {
        "train": {
            "loss": [],
            "loss_classifier": [],
            "loss_box_reg": [],
            "loss_rpn_box_reg": [],
            "loss_objectness": []
        },
        "val": {
            "loss": [],
            "loss_classifier": [],
            "loss_box_reg": [],
            "loss_rpn_box_reg": [],
            "loss_objectness": []
        },
        "map50":{
            "train":[],
            "valid":[],
        }
    }
    best_idx = 0
    print('start')
    for epoch in range(config.epochs):
        print()
        train_loss, train_loss_classifier, train_loss_box_reg, train_loss_rpn_box_reg, train_loss_objectness = train_one_epoch(
            model, train_loader, optimizer, epoch, device,
        )
        
        val_loss, val_loss_classifier, val_loss_box_reg, val_loss_rpn_box_reg, val_loss_objectness = validation(
            model, val_loader, device
        )

        ## Training
        history["train"]["loss"].append(train_loss)
        history["train"]["loss_classifier"].append(train_loss_classifier)
        history["train"]["loss_box_reg"].append(train_loss_box_reg)
        history["train"]["loss_rpn_box_reg"].append(train_loss_rpn_box_reg)
        history["train"]["loss_objectness"].append(train_loss_objectness)
        ## Validation
        history["val"]["loss"].append(val_loss)
        history["val"]["loss_classifier"].append(val_loss_classifier)
        history["val"]["loss_box_reg"].append(val_loss_box_reg)
        history["val"]["loss_rpn_box_reg"].append(val_loss_rpn_box_reg)
        history["val"]["loss_objectness"].append(val_loss_objectness)

        
        print(f'Epoch: {epoch+1}/{config.epochs} | LR: {optimizer.state_dict()["param_groups"][0]["lr"]:.6f}')

        print("*****Training*****")
        print(f'Loss: {train_loss:.4f} | Classifier Loss: {train_loss_classifier:.4f} | Box Reg Loss: {train_loss_box_reg:.4f} | RPN Box Reg Loss: {train_loss_rpn_box_reg:.4f} | Objectness Loss: {train_loss_objectness:.4f}')
        train_evaluator = evaluate(model, train_loader, device = device)
        print("*****Validation*****")
        print(f'Loss: {val_loss:.4f} | Classifier Loss: {val_loss_classifier:.4f} | Box Reg Loss: {val_loss_box_reg:.4f} | RPN Box Reg Loss: {val_loss_rpn_box_reg:.4f} | Objectness Loss: {val_loss_objectness:.4f}')
        valid_evaluator = evaluate(model, val_loader, device = device)
        
        train_map50 = train_evaluator.coco_eval['bbox'].stats[1]
        valid_map50 = valid_evaluator.coco_eval['bbox'].stats[1]
        
        history["map50"]["train"].append(train_map50)   
        history["map50"]["valid"].append(valid_map50)        

        ## TODO save your model

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_file = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
                "args": config
            }
            best_idx = epoch
            torch.save(save_file, os.path.join(config.save_root, "best_model.pth"))
        
    print(f'Best epoch in {best_idx+1}') 



    ## Evaluation result 
    plt.figure(figsize = (12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["map50"]["train"], label = 'Training map50')
    plt.plot(range(config.epochs), history["map50"]["valid"], label = 'Validation map50')
    plt.xlabel('Epoch')
    plt.ylabel('map')
    plt.legend()
    plt.title('Validation and Testing map50')
    plt.show()
    
    plt.figure(figsize = (12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss"], label = 'Training Loss')
    plt.plot(range(config.epochs), history["val"]["loss"], label = 'Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss Curves')
    plt.show()
        
    plt.figure(figsize = (12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss_classifier"], label = 'Training Classifier Loss')
    plt.plot(range(config.epochs), history["val"]["loss_classifier"], label = 'Validation Classifier Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Classifier Loss')
    plt.legend()
    plt.title('Training and Validation Classifier Loss Curves')
    plt.show()
        
    plt.figure(figsize = (12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss_box_reg"], label = 'Training Box Reg Loss')
    plt.plot(range(config.epochs), history["val"]["loss_box_reg"], label = 'Validation Box Reg Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Box Reg Loss')
    plt.legend()
    plt.title('Training and Validation Box Reg Loss Curves')
    plt.show()
        
    plt.figure(figsize = (12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss_rpn_box_reg"], label = 'Training RPN Box Reg Loss')
    plt.plot(range(config.epochs), history["val"]["loss_rpn_box_reg"], label = 'Validation RPN Box Reg Loss')
    
    plt.xlabel('Epoch')
    plt.ylabel('RPN Box Reg Loss')
    plt.legend()
    plt.title('Training and Validation RPN Box Reg Loss Curves')
    plt.show()
        
    plt.figure(figsize = (12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss_objectness"], label = 'Training Objectness Loss')
    plt.plot(range(config.epochs), history["val"]["loss_objectness"], label = 'Validation Objectness Loss')

    plt.xlabel('Epoch')
    plt.ylabel('Objectness Loss')
    plt.legend()
    plt.title('Training and Validation Objectness Loss Curves')
    plt.show()   


In [ ]:
## IMAGENET 3
if __name__ == "__main__":
    main()

## TODO:


In [ ]:
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
class config:
    root = "hwk05_data/output"
    num_classes = 8
    categories = ['normal', 'aortic_curvature', 'aortic_atherosclerosis_calcification', 
                  'cardiac_hypertrophy', 'intercostal_pleural_thickening', 'lung_field_infiltration', 
                  'degenerative_joint_disease_of_the_thoracic_spine', 'scoliosis']
    seed = 42
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = config.device

# 建立模型
model = fasterrcnn_resnet50_fpn(pretrained=False, num_classes=config.num_classes)
model.to(device)

# 加載模型檔案
best_ckpt = torch.load("hwk05_data/best_model.pth", map_location=config.device)
model.load_state_dict(best_ckpt["model"])
model.eval()

# 定義影像變換
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])

# 推論函數
def predict(image_path, model):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(image_tensor)[0]  # 獲取模型預測結果

    boxes = output['boxes']
    scores = output['scores']
    labels = output['labels']

    return boxes, scores, labels

# 讀取測試集檔案
test_csv = 'hwk05_data/test.csv'
test_df = pd.read_csv(test_csv)

# 儲存結果
submission_data = []
test_images_dir = 'hwk05_data/output/test'

for _, row in test_df.iterrows():
    image_id = row['ID']
    original_path = os.path.join(test_images_dir, row['Filename'])

    # 構建 JPG 檔案名稱
    jpg_name = row['Filename'].replace(".dcm", ".jpg")
    image_path = os.path.join(test_images_dir, jpg_name)
    
    boxes, scores, labels = predict(image_path, model)
    image = Image.open(image_path)
    W, H = image.size
    
    for i, box in enumerate(boxes):
        if scores[i] > 0.5 and labels[i].item() != 0:
            xmin, ymin, xmax, ymax = box
            submission_data.append({
                'ID': image_id,
                'category': config.categories[labels[i].item()],
                'score': scores[i].item(),
                'xmin': xmin.item() / W,
                'xmax': xmax.item() / W,
                'ymin': ymin.item() / H,
                'ymax': ymax.item() / H
            })

submission_df = pd.DataFrame(submission_data)
submission_df.to_csv('student.csv', index=False)

print("CSV 文件已生成！")


## TODO2:


In [ ]:
import pandas as pd

print("First 10 rows:")
print(submission_df.head(10))

print("\nLast 10 rows:")
print(submission_df.tail(10))

print("\nShape of the DataFrame:")
print(submission_df.shape)
